# 詩織 LoRA v4e テスト生成ノートブック（Kaggle） — ベースモデル変更テスト

v4d（LoRA 1.2＋ネガティブ）でも目の丸さと輪郭の子供っぽさはほとんど動かなかったため、ベースモデル側を変えるテスト。
学習済みLoRA（train-shiori-lora-kaggle-v4 Version 2 の出力 shiori_oneesan_v4.safetensors）を **Add Input** で接続して使う。学習は行わない。

条件：
- ベースモデルを Animagine XL 4.0 から **NoobAI-XL 1.1**（Illustrious系、大人顔寄り・仕上がりが綺麗とされる系統）に変更。取得できなければ Illustrious XL 本家にフォールバック
- プロンプト・ネガティブ・シード（42）・28ステップ・LoRA scale 1.0 は v4 と同一にして、変えるのはベースモデルだけ
- 生成は2枚（商店街カフェ／髪のアップ）

In [ ]:
# GPU確認
!nvidia-smi


In [ ]:
# 依存関係の用意（torchao は古い版が入っていると peft が ImportError になるため外す）
!pip install -q diffusers transformers accelerate peft huggingface_hub
!pip uninstall -y torchao

# ベースモデルを取得（NoobAI-XL 1.1 を第一候補、ダメなら Illustrious XL 本家）
from huggingface_hub import hf_hub_download, HfApi
import shutil, os
api = HfApi()
candidates = ['Laxhar/noobai-XL-1.1', 'OnomaAIResearch/Illustrious-xl-early-release-v0']
base_path = None
for repo in candidates:
    try:
        files = [f for f in api.list_repo_files(repo) if f.endswith('.safetensors')]
        print(repo, 'safetensors:', files)
        # VAEやテキストエンコーダ単体っぽい名前を避け、いちばん大きい本命ファイルを選ぶ
        main_files = [f for f in files if 'vae' not in f.lower() and 'text_encoder' not in f.lower() and 'clip' not in f.lower()]
        fname = main_files[0] if main_files else files[0]
        p = hf_hub_download(repo_id=repo, filename=fname)
        shutil.copy(p, '/kaggle/working/base_model.safetensors')
        base_path = '/kaggle/working/base_model.safetensors'
        print('base model ready:', repo, fname, os.path.getsize(base_path))
        break
    except Exception as e:
        print('failed for', repo, ':', e)
assert base_path, 'no base model could be downloaded'


In [ ]:
# LoRA を読み込んでテスト生成（2枚、条件は v4 と同一）
import glob, os, torch
from diffusers import StableDiffusionXLPipeline
cands = [c for c in glob.glob('/kaggle/input/**/shiori_oneesan_v4.safetensors', recursive=True)]
print('lora candidates:', cands)
assert cands, 'shiori_oneesan_v4.safetensors not found in /kaggle/input'
lora_path = cands[0]
print('lora size:', os.path.getsize(lora_path))
pipe = StableDiffusionXLPipeline.from_single_file('/kaggle/working/base_model.safetensors', torch_dtype=torch.float16).to('cuda')
pipe.load_lora_weights(os.path.dirname(lora_path), weight_name='shiori_oneesan_v4.safetensors')
neg = 'lowres, bad anatomy, bad hands, extra fingers, worst quality, blurry, multiple girls, 2girls, extra person, tsurime, sharp eyes, narrow eyes'
os.makedirs('/kaggle/working/output', exist_ok=True)

prompt1 = 'shiori_oneesan, 1woman, solo, very long hair, straight black hair, matte black hair, hime cut, blunt bangs, side part, purple eyes, gentle eyes, droopy eyes, tareme, gentle smile, soft expression, pink cardigan, shopping street, cafe terrace, sunny day'
g = torch.Generator(device='cuda').manual_seed(42)
img = pipe(prompt1, negative_prompt=neg, width=1024, height=1024, num_inference_steps=28, guidance_scale=7, generator=g).images[0]
img.save('/kaggle/working/output/test_outing_v4e.png')
print('saved test_outing_v4e.png')

prompt2 = 'shiori_oneesan, 1woman, solo, very long hair, straight black hair, matte black hair, hime cut, blunt bangs, side part, purple eyes, gentle eyes, droopy eyes, tareme, gentle smile, soft expression, hair over shoulder, white blouse, indoors, window, daylight, close-up portrait'
g = torch.Generator(device='cuda').manual_seed(42)
img2 = pipe(prompt2, negative_prompt=neg, width=1024, height=1024, num_inference_steps=28, guidance_scale=7, generator=g).images[0]
img2.save('/kaggle/working/output/test_hair_v4e.png')
print('saved test_hair_v4e.png')
img
